# Theta power and movement

This notebook plots the theta time-frequency spetrogram and movement trajectory to check alignment of neural and behavioral data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ShortTimeFFT
import os

In [2]:
DATA_DIR = r"\\research-cifs.nyumc.org\research\buzsakilab\Homes\voerom01\Bilat_HPC\Bilat_R02\Bilat_R02_20251106"
PROBE = 3
CHANNEL = 163

In [10]:
session_name = os.path.basename(DATA_DIR)
print(f"Processing session: {session_name}")
dat_file = os.path.join(DATA_DIR, f"{session_name}_imec{PROBE}.dat")
xml_file = os.path.join(DATA_DIR, f"{session_name}_imec{PROBE}.xml")

#getting number of channels from xml
import xml.etree.ElementTree as ET
tree = ET.parse(xml_file)
root = tree.getroot()
nChannels = int(root.find('acquisitionSystem').find('nChannels').text)
samplingRate = float(root.find('acquisitionSystem').find('samplingRate').text)
nBits = int(root.find('acquisitionSystem').find('nBits').text)
amplification = float(root.find('acquisitionSystem').find('amplification').text)
print(f"Sampling rate: {samplingRate} Hz")
print(f"Number of channels: {nChannels}")
print(f"Number of bits: {nBits}")
print(f"Amplification: {amplification}")

# loading the dat file
data = np.memmap(dat_file, dtype='int16', mode='r')
data = data.reshape(-1, nChannels)
channel_data = data[:, CHANNEL] / amplification  # Convert to microvolts
print(f"Loaded data shape: {channel_data.shape}")


Processing session: Bilat_R02_20251106
Sampling rate: 30000.0 Hz
Number of channels: 385
Number of bits: 16
Amplification: 1000.0
Loaded data shape: (315630436,)


In [ ]:
# plotting the raw data for the selected channel
plt.figure(figsize=(15, 5))
plt.plot(channel_data[:samplingRate * 10])  # Plot first 10 seconds
plt.title(f"Raw Data - Channel {CHANNEL}")
plt.xticks(np.arange(0, samplingRate * 10, samplingRate), np.arange(0, 11, 1))
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (µV)")
plt.show()


In [ ]:
win = np.hanning(int(samplingRate * 0.5))  # 0.5 second window
SFT = ShortTimeFFT(win, fs=samplingRate, mfft = 200, scale_to='magnitude')
Sx = SFT.stft(channel_data)

In [ ]:
N = len(channel_data)

fig1, ax1 = plt.subplots(figsize=(6., 4.))  # enlarge plot a bit
t_lo, t_hi = SFT.extent(N)[:2]  # time range of plot
ax1.set_title(rf"STFT ({SFT.m_num*SFT.T:g}$\,s$ hanning window)")
ax1.set(xlabel=f"Time $t$ in seconds ({SFT.p_num(N)} slices, " +
               rf"$\Delta t = {SFT.delta_t:g}\,$s)",
        ylabel=f"Freq. $f$ in Hz ({SFT.f_pts} bins, " +
               rf"$\Delta f = {SFT.delta_f:g}\,$Hz)",
        xlim=(t_lo, t_hi))

im1 = ax1.imshow(abs(Sx), origin='lower', aspect='auto',
                 extent=SFT.extent(N), cmap='viridis')

fig1.colorbar(im1, label="Magnitude $|S_x(t, f)|$")

# Shade areas where window slices stick out to the side:
for t0_, t1_ in [(t_lo, SFT.lower_border_end[0] * SFT.T),
                 (SFT.upper_border_begin(N)[0] * SFT.T, t_hi)]:
    ax1.axvspan(t0_, t1_, color='w', linewidth=0, alpha=.2)
for t_ in [0, N * SFT.T]:  # mark signal borders with vertical line:
    ax1.axvline(t_, color='y', linestyle='--', alpha=0.5)
ax1.legend()
fig1.tight_layout()
plt.show()